In [ ]:
!pip install rembg[gpu] pillow


In [ ]:
import os
import io
from google.colab import files
from rembg import remove, new_session
from PIL import Image
import IPython.display as display # Նկարները ցուցադրելու համար

# 1. Կարգավորումներ
TARGET_SIZE = 1000
input_folder = 'demo_input'
output_folder = 'demo_output'
os.makedirs(input_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)
session = new_session("u2net")

# 2. Վերբեռնում
print("📸 Step 1: Upload your product photo")
uploaded = files.upload()

for filename in uploaded.keys():
    input_path = os.path.join(input_folder, filename)
    with open(input_path, 'wb') as f:
        f.write(uploaded[filename])

    # --- ՄՇԱԿՈՒՄ ---
    output_path = os.path.join(output_folder, os.path.splitext(filename)[0] + "_final.jpg")

    with open(input_path, 'rb') as i:
        img_data = i.read()
        nobg_data = remove(img_data, session=session)
        nobg_img = Image.open(io.BytesIO(nobg_data)).convert("RGBA")

        bbox = nobg_img.getbbox()
        if bbox: nobg_img = nobg_img.crop(bbox)

        final_canvas = Image.new("RGBA", (TARGET_SIZE, TARGET_SIZE), (255, 255, 255, 255))
        nobg_img.thumbnail((900, 900), Image.Resampling.LANCZOS)
        offset = ((TARGET_SIZE - nobg_img.width) // 2, (TARGET_SIZE - nobg_img.height) // 2)
        final_canvas.paste(nobg_img, offset, nobg_img)
        final_canvas.convert("RGB").save(output_path, "JPEG", quality=95)

    # --- ՎԻԶՈՒԱԼԻԶԱՑԻԱ (Սա այն մասն է, որը հաճախորդը կտեսնի) ---
    print(f"\n✨ Comparison for: {filename}")

    # Ստեղծում ենք Before/After տեսք
    print("BEFORE (Original) vs AFTER (1000x1000 White BG)")

    # Ցուցադրում ենք երկու նկարն էլ կողք-կողքի (փոքրացված տեսքով)
    original_display = Image.open(input_path)
    original_display.thumbnail((300, 300))

    final_display = Image.open(output_path)
    final_display.thumbnail((300, 300))

    # Ցուցադրում ենք էկրանին
    display.display(original_display)
    display.display(final_display)
    print("-" * 50)

    # Ավտոմատ ներբեռնում
    files.download(output_path)